# Experiment 3.0.4 — L2 width and representation capacity

Analysis-only notebook. Training/evaluation are performed by the Slurm pipeline.

Fixed architecture: `30 -> L1 128 (2,3,4) -> L2 W (2,3,4) -> readout`, with `W in {32,64,128,256}` and no L3. W128 reuses Experiment 3.0.3-B.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('artifacts/experiment_3_0_4_l2_width_representation_capacity/l2_width_v1')
native = pd.read_csv(ROOT / 'experiment_3_0_4_native_summary.csv')
probes = pd.read_csv(ROOT / 'experiment_3_0_4_layer_probe_summary.csv')
diagnostics = pd.read_csv(ROOT / 'experiment_3_0_4_diagnostic_summary.csv')
firing = pd.read_csv(ROOT / 'experiment_3_0_4_firing_rate_summary.csv')


## Native balanced accuracy vs L2 width


In [ ]:
display(native[['objective', 'l2_width', 'mean_test_balanced_accuracy', 'sd_test_balanced_accuracy', 'mean_train_test_ba_gap', 'mean_head_feature_dim', 'mean_head_parameter_count']])

fig, ax = plt.subplots(figsize=(8, 5))
for objective, part in native.groupby('objective'):
    part = part.sort_values('l2_width')
    ax.errorbar(part['l2_width'], part['mean_test_balanced_accuracy'], yerr=part['sd_test_balanced_accuracy'], marker='o', label=objective)
ax.set_xlabel('L2 width')
ax.set_ylabel('Test balanced accuracy')
ax.set_title('Native performance vs L2 width')
ax.legend()
plt.show()


## Raw fixed250 vs PCA128 dimension-matched probe

If raw probe BA continues increasing with width but PCA128 saturates, the extra gain is likely associated with the growing downstream feature dimension rather than a more compact discriminative representation.


In [ ]:
l2 = probes[(probes['layer'] == 'L2') & (probes['probe_type'].isin(['fixed250', 'fixed250_pca128']))].copy()
display(l2[['objective', 'l2_width', 'probe_type', 'mean_input_feature_dim', 'mean_feature_dim', 'mean_probe_test_balanced_accuracy', 'sd_probe_test_balanced_accuracy']])

for objective, part in l2.groupby('objective'):
    fig, ax = plt.subplots(figsize=(8, 5))
    for probe_type, probe_part in part.groupby('probe_type'):
        probe_part = probe_part.sort_values('l2_width')
        ax.errorbar(probe_part['l2_width'], probe_part['mean_probe_test_balanced_accuracy'], yerr=probe_part['sd_probe_test_balanced_accuracy'], marker='o', label=probe_type)
    ax.set_xlabel('L2 width')
    ax.set_ylabel('Frozen probe test BA')
    ax.set_title(objective)
    ax.legend()
    plt.show()


## L1 -> L2 representation gain

The PCA128 gain is the cleaner comparison because both L1 and L2 are projected to the same dimension.


In [ ]:
display(diagnostics[['objective', 'l2_width', 'mean_raw_l2_minus_l1_fixed250_probe_ba', 'mean_pca128_l2_minus_l1_fixed250_probe_ba', 'mean_l2_raw_minus_pca128_fixed250_probe_ba', 'mean_l2_pca128_explained_variance_ratio_sum']])

fig, ax = plt.subplots(figsize=(8, 5))
for objective, part in diagnostics.groupby('objective'):
    part = part.sort_values('l2_width')
    ax.plot(part['l2_width'], part['mean_pca128_l2_minus_l1_fixed250_probe_ba'], marker='o', label=objective)
ax.axhline(0.0, linewidth=1)
ax.set_xlabel('L2 width')
ax.set_ylabel('PCA128 L2 - L1 probe BA')
ax.set_title('Dimension-matched representation gain')
ax.legend()
plt.show()


## Activity / cost trade-off


In [ ]:
cost = diagnostics[['objective', 'l2_width', 'mean_test_balanced_accuracy', 'mean_l2_test_firing_rate', 'mean_l2_test_total_spikes_per_timestep']].copy()
display(cost)

fig, ax = plt.subplots(figsize=(8, 5))
for objective, part in cost.groupby('objective'):
    part = part.sort_values('l2_width')
    ax.plot(part['l2_width'], part['mean_l2_test_total_spikes_per_timestep'], marker='o', label=objective)
ax.set_xlabel('L2 width')
ax.set_ylabel('L2 total spikes / valid timestep')
ax.set_title('Spike activity cost vs L2 width')
ax.legend()
plt.show()
